# 10. Creating a choropleth map

A **choropleth** shades geographic areas according to a numeric variable.

The geometry tells GeoPandas **where to draw** each ward.

The selected numeric column tells GeoPandas **how to shade** it.

## The joined GeoDataFrame makes mapping possible

After the join, each row contains both geography and statistics:

| Ward | library_count | total_workstations | geometry |
|---|---:|---:|---|
| Ward A | 4 | 62 | POLYGON(...) |
| Ward B | 2 | 31 | POLYGON(...) |
| Ward C | 5 | 80 | POLYGON(...) |

We can therefore map different variables using the same ward polygons.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

libraries = pd.read_csv(
    "../data/tpl-branch-general-information-2023.csv"
)

wards = gpd.read_file(
    "../data/city-wards.gpkg"
)

## Recreate the ward summary and join

This notebook repeats the preparation so it can run independently.

In [ ]:
physical_libraries = libraries.loc[
    libraries["PhysicalBranch"] == 1
].copy()

ward_summary = (
    physical_libraries
    .groupby("WardNo")
    .agg(
        library_count=("BranchName", "count"),
        total_workstations=("Workstations", "sum")
    )
    .reset_index()
)

ward_summary["WARD_SHORT_CODE"] = (
    ward_summary["WardNo"]
    .astype(int)
    .astype(str)
    .str.zfill(2)
)

wards_with_libraries = wards.merge(
    ward_summary,
    on="WARD_SHORT_CODE",
    how="left"
)

## Raw totals versus normalized measures

A raw count asks:

> How many libraries are located in each ward?

A normalized measure can ask:

> How concentrated are libraries relative to ward area?

| Ward | Libraries | Area | Libraries per 10 km² |
|---|---:|---:|---:|
| A | 4 | 10 km² | 4.0 |
| B | 4 | 40 km² | 1.0 |

Both wards contain four libraries, but their spatial concentration differs.

In [ ]:
wards_with_libraries["area_km2"] = (
    wards_with_libraries.geometry.area
    / 1_000_000
)

wards_with_libraries["libraries_per_10_km2"] = (
    wards_with_libraries["library_count"]
    / wards_with_libraries["area_km2"]
    * 10
)

## Map 1: total library branches

This emphasizes **absolute totals**.

In [ ]:
ax = wards_with_libraries.plot(
    column="library_count",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    figsize=(8, 8)
)

ax.set_title("Toronto Public Library branches by ward")
ax.set_axis_off()

## Map 2: libraries per 10 km²

This emphasizes **spatial concentration**.

In [ ]:
ax = wards_with_libraries.plot(
    column="libraries_per_10_km2",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    figsize=(8, 8)
)

ax.set_title("Toronto Public Libraries per 10 km², by ward")
ax.set_axis_off()

## What changes when we normalize?

Two maps can use the same source data but answer different questions.

| Measure | Best interpreted as |
|---|---|
| `library_count` | absolute number of branches |
| `libraries_per_10_km2` | spatial concentration of branches |

Neither automatically measures accessibility or equity.

## Choropleth checklist

Before interpreting a choropleth, ask:

1. What does one polygon represent?
2. What numeric variable controls the shading?
3. Is a raw count appropriate, or should it be normalized?
4. Are any values missing?
5. Does the title clearly describe the measure?
6. Could differences in geographic size affect interpretation?

For library access specifically, we might eventually also want population, travel distance, transit access, branch size, capacity, or measures of community need.